## Model Comparison for Li-CO2 and Li-O2 Overpotential Prediction

This notebook compares eight regressors using iteration 6 datasets:
XGBoost, Ridge, Kernel Ridge, SVR, Gaussian Process, Random Forest,
Gradient Boosting, and TabPFN.

### Workflow

1. Load each dataset and retain all input features.
2. Split by compound, reserving 20% of compound groups for testing.
3. Evaluate model configurations using five-fold grouped
   cross-validation within the training set.
4. Select the best configuration for each model by CV MAE and
   evaluate training and test R², MAE, and RMSE.
5. Export test predictions and comparison metrics ranked by CV R².

TabPFN runs on CUDA using the specified local checkpoint.
The CO2 and O2 sections can run independently. Complete each section
in order, including the comparison export, before switching systems.

## CO2

In [1]:
# ============================================================
# 1. Import packages
# ============================================================

import os
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from tabpfn import TabPFNRegressor

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.gaussian_process import (
    GaussianProcessRegressor
)
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    GroupShuffleSplit
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

In [2]:
# ============================================================
# 2. Basic settings
# ============================================================

DATA_FILE = "data/compound_6_CO2.csv"

TARGET_COL = "overpotential"
GROUP_COL = "compound"

TEST_SIZE = 0.20
RANDOM_STATE = 888

TABPFN_MODEL_PATH = (
    "tabpfn_models/"
    "tabpfn-v3-regressor-v3_default.ckpt"
)
OUTPUT_DIR = "model_comparison/CO2"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [3]:
# ============================================================
# 3. Load data
# ============================================================

data = pd.read_csv(DATA_FILE)

data.columns = (
    data.columns
    .astype(str)
    .str.strip()
)

# Remove only automatically generated index columns
data = data.drop(
    columns=[
        col for col in data.columns
        if col.startswith("Unnamed:")
    ],
    errors="ignore"
)

GROUP_COL = (
    "compound"
    if "compound" in data.columns
    else "compounds"
)

feature_cols = [
    col for col in data.columns
    if col not in [GROUP_COL, TARGET_COL]
]

X = data[feature_cols]
y = data[TARGET_COL]
groups = data[GROUP_COL]

print("Data columns:", data.columns.tolist())
print("Group column:", GROUP_COL)
print("Target column:", TARGET_COL)
print("Total samples:", len(data))
print("Total compounds:", groups.nunique())
print("Number of features:", len(feature_cols))
print("Feature columns:", feature_cols)

Data columns: ['compound', 'AB_AN', 'AB_AW', 'AB_CR', 'AB_AR', 'AB_FIE', 'AB_SIE', 'AB_EA', 'AB_EN', 'AB_Nd', 'AB_Nv', 'AB_D', 'AB_EC', 'AB_TC', 'C_AN', 'C_AW', 'C_CR', 'C_AR', 'C_VR', 'C_FIE', 'C_SIE', 'C_EA', 'C_EN', 'C_Np', 'C_Nv', 'C_D', 'a_1', 'b_1', 'c_1', 'Grain_1', 'a_2', 'b_2', 'c_2', 'Grain_2', 'CurrentDensity', 'overpotential']
Group column: compound
Target column: overpotential
Total samples: 708
Total compounds: 122
Number of features: 34
Feature columns: ['AB_AN', 'AB_AW', 'AB_CR', 'AB_AR', 'AB_FIE', 'AB_SIE', 'AB_EA', 'AB_EN', 'AB_Nd', 'AB_Nv', 'AB_D', 'AB_EC', 'AB_TC', 'C_AN', 'C_AW', 'C_CR', 'C_AR', 'C_VR', 'C_FIE', 'C_SIE', 'C_EA', 'C_EN', 'C_Np', 'C_Nv', 'C_D', 'a_1', 'b_1', 'c_1', 'Grain_1', 'a_2', 'b_2', 'c_2', 'Grain_2', 'CurrentDensity']


In [4]:
# ============================================================
# 4. Group-based train/test split
# ============================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_index, test_index = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

train_data = (
    data
    .iloc[train_index]
    .copy()
    .reset_index(drop=True)
)

test_data = (
    data
    .iloc[test_index]
    .copy()
    .reset_index(drop=True)
)

X_train = train_data[feature_cols]
X_test = test_data[feature_cols]

y_train = (
    train_data[TARGET_COL]
    .to_numpy()
)

y_test = (
    test_data[TARGET_COL]
    .to_numpy()
)

groups_train = (
    train_data[GROUP_COL]
    .to_numpy()
)

print("\nTraining samples:", len(train_data))
print("Test samples:", len(test_data))
print(
    "Training compounds:",
    train_data[GROUP_COL].nunique()
)
print(
    "Test compounds:",
    test_data[GROUP_COL].nunique()
)

overlap = (
    set(train_data[GROUP_COL])
    & set(test_data[GROUP_COL])
)

print("Overlapping compounds:", len(overlap))



Training samples: 557
Test samples: 151
Training compounds: 97
Test compounds: 25
Overlapping compounds: 0


In [5]:
# ============================================================
# 5. Group cross-validation
# ============================================================

cv = GroupKFold(
    n_splits=5
)

scoring = {
    "R2": "r2",
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error"
}


# ============================================================
# 6. Model training and evaluation function
# ============================================================

def train_and_evaluate(
    model_name,
    model,
    param_grid,
    n_jobs=-1
):

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=scoring,
        cv=cv,
        n_jobs=n_jobs,
        refit="MAE",
        return_train_score=False
    )

    search.fit(
        X_train,
        y_train,
        groups=groups_train
    )

    best_model = search.best_estimator_
    best_index = search.best_index_

    cv_r2 = (
        search.cv_results_[
            "mean_test_R2"
        ][best_index]
    )

    cv_mae = -(
        search.cv_results_[
            "mean_test_MAE"
        ][best_index]
    )

    cv_rmse = -(
        search.cv_results_[
            "mean_test_RMSE"
        ][best_index]
    )

    train_prediction = best_model.predict(
        X_train
    )

    test_prediction = best_model.predict(
        X_test
    )

    train_r2 = r2_score(
        y_train,
        train_prediction
    )

    train_mae = mean_absolute_error(
        y_train,
        train_prediction
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_train,
            train_prediction
        )
    )

    test_r2 = r2_score(
        y_test,
        test_prediction
    )

    test_mae = mean_absolute_error(
        y_test,
        test_prediction
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test_prediction
        )
    )

    print(f"\n===== {model_name} =====")
    print("Best parameters:", search.best_params_)

    print(
        f"CV: R2={cv_r2:.6f} | "
        f"MAE={cv_mae:.6f} | "
        f"RMSE={cv_rmse:.6f}"
    )

    print(
        f"Train: R2={train_r2:.6f} | "
        f"MAE={train_mae:.6f} | "
        f"RMSE={train_rmse:.6f}"
    )

    print(
        f"Test: R2={test_r2:.6f} | "
        f"MAE={test_mae:.6f} | "
        f"RMSE={test_rmse:.6f}"
    )

    result = {
        "Model": model_name,
        "CV_R2": cv_r2,
        "CV_MAE": cv_mae,
        "CV_RMSE": cv_rmse,
        "Train_R2": train_r2,
        "Train_MAE": train_mae,
        "Train_RMSE": train_rmse,
        "Test_R2": test_r2,
        "Test_MAE": test_mae,
        "Test_RMSE": test_rmse,
        "Best_parameters": str(
            search.best_params_
        )
    }

    prediction_result = pd.DataFrame({
        GROUP_COL: test_data[GROUP_COL],
        "y_true": y_test,
        "y_pred": test_prediction
    })

    prediction_result.to_csv(
        f"{OUTPUT_DIR}/{model_name}_test_predictions.csv",
        index=False
    )

    return best_model, result


In [6]:
# ============================================================
# 7. XGBoost
# ============================================================

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

xgb_grid = {
    "n_estimators": [
        100,
        200,
        300
    ],
    "max_depth": [
        2,
        3,
        4
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ]
}

best_xgb, xgb_result = train_and_evaluate(
    model_name="XGBoost",
    model=xgb_model,
    param_grid=xgb_grid
)



===== XGBoost =====
Best parameters: {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 300}
CV: R2=0.734967 | MAE=0.167426 | RMSE=0.219604
Train: R2=0.949777 | MAE=0.070323 | RMSE=0.097015
Test: R2=0.804715 | MAE=0.123998 | RMSE=0.148835


In [7]:
# ============================================================
# 8. Ridge
# ============================================================

ridge_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        Ridge()
    )
])

ridge_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ]
}

best_ridge, ridge_result = train_and_evaluate(
    model_name="Ridge",
    model=ridge_model,
    param_grid=ridge_grid
)



===== Ridge =====
Best parameters: {'model__alpha': 0.1}
CV: R2=0.723478 | MAE=0.173157 | RMSE=0.224233
Train: R2=0.813450 | MAE=0.144562 | RMSE=0.186976
Test: R2=0.628161 | MAE=0.157773 | RMSE=0.205375


In [8]:
# ============================================================
# 9. Kernel ridge
# ============================================================

krr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        KernelRidge(
            kernel="rbf"
        )
    )
])

krr_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10
    ],
    "model__gamma": [
        0.001,
        0.01,
        0.1,
        1
    ]
}

best_krr, krr_result = train_and_evaluate(
    model_name="Kernel Ridge",
    model=krr_model,
    param_grid=krr_grid
)



===== Kernel Ridge =====
Best parameters: {'model__alpha': 0.01, 'model__gamma': 0.001}
CV: R2=0.743594 | MAE=0.166550 | RMSE=0.215578
Train: R2=0.830833 | MAE=0.137839 | RMSE=0.178051
Test: R2=0.769039 | MAE=0.123659 | RMSE=0.161860


In [9]:
# ============================================================
# 10. Support vector regression
# ============================================================

svr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        SVR(
            kernel="rbf"
        )
    )
])

svr_grid = {
    "model__C": [
        0.1,
        1,
        10,
        100
    ],
    "model__epsilon": [
        0.01,
        0.05,
        0.10,
        0.20
    ],
    "model__gamma": [
        "scale",
        0.01,
        0.10,
        1.0
    ]
}

best_svr, svr_result = train_and_evaluate(
    model_name="SVR",
    model=svr_model,
    param_grid=svr_grid
)


===== SVR =====
Best parameters: {'model__C': 10, 'model__epsilon': 0.01, 'model__gamma': 0.01}
CV: R2=0.708432 | MAE=0.167495 | RMSE=0.228751
Train: R2=0.866880 | MAE=0.106711 | RMSE=0.157946
Test: R2=0.852787 | MAE=0.096130 | RMSE=0.129224


In [10]:
# ============================================================
# 11. Gaussian process regression
# ============================================================

gpr_kernel = (
    ConstantKernel(
        constant_value=1.0,
        constant_value_bounds=(1e-2, 1e2)
    )
    * RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-2, 1e2)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e0)
    )
)

gpr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        GaussianProcessRegressor(
            kernel=gpr_kernel,
            normalize_y=True,
            n_restarts_optimizer=5,
            random_state=RANDOM_STATE
        )
    )
])

gpr_grid = {
    "model__alpha": [
        1e-8,
        1e-6,
        1e-4,
        1e-2
    ]
}

best_gpr, gpr_result = train_and_evaluate(
    model_name="GPR",
    model=gpr_model,
    param_grid=gpr_grid
)


===== GPR =====
Best parameters: {'model__alpha': 1e-06}
CV: R2=0.639257 | MAE=0.186920 | RMSE=0.252027
Train: R2=0.925578 | MAE=0.085778 | RMSE=0.118096
Test: R2=0.857331 | MAE=0.101158 | RMSE=0.127214


In [11]:
# ============================================================
# 12. Random forest
# ============================================================

rf_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=1
)

rf_grid = {
    "n_estimators": [
        100,
        300,
        500
    ],
    "max_depth": [
        None,
        5,
        10
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_rf, rf_result = train_and_evaluate(
    model_name="Random Forest",
    model=rf_model,
    param_grid=rf_grid
)



===== Random Forest =====
Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 300}
CV: R2=0.720654 | MAE=0.167824 | RMSE=0.223636
Train: R2=0.976691 | MAE=0.041419 | RMSE=0.066092
Test: R2=0.851446 | MAE=0.106355 | RMSE=0.129811


In [12]:
# ============================================================
# 13. Gradient boosting
# ============================================================

gbdt_model = GradientBoostingRegressor(
    random_state=RANDOM_STATE
)

gbdt_grid = {
    "n_estimators": [
        100,
        200,
        300
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ],
    "max_depth": [
        1,
        2,
        3
    ],
    "min_samples_leaf": [
        2,
        4,
        6
    ]
}

best_gbdt, gbdt_result = train_and_evaluate(
    model_name="Gradient Boosting",
    model=gbdt_model,
    param_grid=gbdt_grid
)


===== Gradient Boosting =====
Best parameters: {'learning_rate': 0.05, 'max_depth': 2, 'min_samples_leaf': 6, 'n_estimators': 300}
CV: R2=0.752993 | MAE=0.166892 | RMSE=0.213333
Train: R2=0.925052 | MAE=0.089496 | RMSE=0.118513
Test: R2=0.792806 | MAE=0.121787 | RMSE=0.153306


In [13]:
# ============================================================
# 14. TabPFN
# ============================================================

tabpfn_model = TabPFNRegressor(
    device="cuda",
    model_path=TABPFN_MODEL_PATH,
    random_state=RANDOM_STATE
)

tabpfn_grid = {}

best_tabpfn, tabpfn_result = train_and_evaluate(
    model_name="TabPFN",
    model=tabpfn_model,
    param_grid=tabpfn_grid,
    n_jobs=1
)


===== TabPFN =====
Best parameters: {}
CV: R2=0.792009 | MAE=0.152364 | RMSE=0.196143
Train: R2=0.985837 | MAE=0.023280 | RMSE=0.051518
Test: R2=0.839428 | MAE=0.103751 | RMSE=0.134960


In [14]:
# ============================================================
# 15. Model comparison
# ============================================================

comparison_results = pd.DataFrame([
    xgb_result,
    ridge_result,
    krr_result,
    svr_result,
    gpr_result,
    rf_result,
    gbdt_result,
    tabpfn_result
])

comparison_results = (
    comparison_results
    .sort_values(
        "CV_R2",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nFinal model comparison:")

print(
    comparison_results[
        [
            "Model",
            "CV_R2",
            "CV_MAE",
            "CV_RMSE",
            "Train_R2",
            "Train_MAE",
            "Train_RMSE",
            "Test_R2",
            "Test_MAE",
            "Test_RMSE"
        ]
    ]
)

comparison_results.to_csv(
    f"{OUTPUT_DIR}/model_comparison_results.csv",
    index=False
)



Final model comparison:
               Model     CV_R2    CV_MAE   CV_RMSE  Train_R2  Train_MAE  \
0             TabPFN  0.792009  0.152364  0.196143  0.985837   0.023280   
1  Gradient Boosting  0.752993  0.166892  0.213333  0.925052   0.089496   
2       Kernel Ridge  0.743594  0.166550  0.215578  0.830833   0.137839   
3            XGBoost  0.734967  0.167426  0.219604  0.949777   0.070323   
4              Ridge  0.723478  0.173157  0.224233  0.813450   0.144562   
5      Random Forest  0.720654  0.167824  0.223636  0.976691   0.041419   
6                SVR  0.708432  0.167495  0.228751  0.866880   0.106711   
7                GPR  0.639257  0.186920  0.252027  0.925578   0.085778   

   Train_RMSE   Test_R2  Test_MAE  Test_RMSE  
0    0.051518  0.839428  0.103751   0.134960  
1    0.118513  0.792806  0.121787   0.153306  
2    0.178051  0.769039  0.123659   0.161860  
3    0.097015  0.804715  0.123998   0.148835  
4    0.186976  0.628161  0.157773   0.205375  
5    0.066092  0.

## O2

In [1]:
# ============================================================
# 1. Import packages
# ============================================================

import os
import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from tabpfn import TabPFNRegressor

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor
)
from sklearn.gaussian_process import (
    GaussianProcessRegressor
)
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF,
    WhiteKernel
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import (
    GridSearchCV,
    GroupKFold,
    GroupShuffleSplit
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

In [2]:
# ============================================================
# 2. Basic settings
# ============================================================

DATA_FILE = "data/compound_6_O2.csv"

TARGET_COL = "overpotential"
GROUP_COL = "compound"

TEST_SIZE = 0.20
RANDOM_STATE = 2000

TABPFN_MODEL_PATH = (
    "tabpfn_models/"
    "tabpfn-v3-regressor-v3_default.ckpt"
)
OUTPUT_DIR = "model_comparison/O2"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

In [3]:
# ============================================================
# 3. Load data
# ============================================================

data = pd.read_csv(DATA_FILE)

data.columns = (
    data.columns
    .astype(str)
    .str.strip()
)

# Remove only automatically generated index columns
data = data.drop(
    columns=[
        col for col in data.columns
        if col.startswith("Unnamed:")
    ],
    errors="ignore"
)

GROUP_COL = (
    "compound"
    if "compound" in data.columns
    else "compounds"
)

feature_cols = [
    col for col in data.columns
    if col not in [GROUP_COL, TARGET_COL]
]

X = data[feature_cols]
y = data[TARGET_COL]
groups = data[GROUP_COL]

print("Data columns:", data.columns.tolist())
print("Group column:", GROUP_COL)
print("Target column:", TARGET_COL)
print("Total samples:", len(data))
print("Total compounds:", groups.nunique())
print("Number of features:", len(feature_cols))
print("Feature columns:", feature_cols)

Data columns: ['compound', 'AB_AN', 'AB_AW', 'AB_CR', 'AB_AR', 'AB_FIE', 'AB_SIE', 'AB_EA', 'AB_EN', 'AB_Nd', 'AB_Nv', 'AB_D', 'AB_EC', 'AB_TC', 'C_AN', 'C_AW', 'C_CR', 'C_AR', 'C_VR', 'C_FIE', 'C_SIE', 'C_EA', 'C_EN', 'C_Np', 'C_Nv', 'C_D', 'a_1', 'b_1', 'c_1', 'Grain_1', 'a_2', 'b_2', 'c_2', 'Grain_2', 'CurrentDensity', 'overpotential']
Group column: compound
Target column: overpotential
Total samples: 449
Total compounds: 97
Number of features: 34
Feature columns: ['AB_AN', 'AB_AW', 'AB_CR', 'AB_AR', 'AB_FIE', 'AB_SIE', 'AB_EA', 'AB_EN', 'AB_Nd', 'AB_Nv', 'AB_D', 'AB_EC', 'AB_TC', 'C_AN', 'C_AW', 'C_CR', 'C_AR', 'C_VR', 'C_FIE', 'C_SIE', 'C_EA', 'C_EN', 'C_Np', 'C_Nv', 'C_D', 'a_1', 'b_1', 'c_1', 'Grain_1', 'a_2', 'b_2', 'c_2', 'Grain_2', 'CurrentDensity']


In [4]:
# ============================================================
# 4. Group-based train/test split
# ============================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

train_index, test_index = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

train_data = (
    data
    .iloc[train_index]
    .copy()
    .reset_index(drop=True)
)

test_data = (
    data
    .iloc[test_index]
    .copy()
    .reset_index(drop=True)
)

X_train = train_data[feature_cols]
X_test = test_data[feature_cols]

y_train = (
    train_data[TARGET_COL]
    .to_numpy()
)

y_test = (
    test_data[TARGET_COL]
    .to_numpy()
)

groups_train = (
    train_data[GROUP_COL]
    .to_numpy()
)

print("\nTraining samples:", len(train_data))
print("Test samples:", len(test_data))
print(
    "Training compounds:",
    train_data[GROUP_COL].nunique()
)
print(
    "Test compounds:",
    test_data[GROUP_COL].nunique()
)

overlap = (
    set(train_data[GROUP_COL])
    & set(test_data[GROUP_COL])
)

print("Overlapping compounds:", len(overlap))



Training samples: 352
Test samples: 97
Training compounds: 77
Test compounds: 20
Overlapping compounds: 0


In [5]:
# ============================================================
# 5. Group cross-validation
# ============================================================

cv = GroupKFold(
    n_splits=5
)

scoring = {
    "R2": "r2",
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error"
}


# ============================================================
# 6. Model training and evaluation function
# ============================================================

def train_and_evaluate(
    model_name,
    model,
    param_grid,
    n_jobs=-1
):

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring=scoring,
        cv=cv,
        n_jobs=n_jobs,
        refit="MAE",
        return_train_score=False
    )

    search.fit(
        X_train,
        y_train,
        groups=groups_train
    )

    best_model = search.best_estimator_
    best_index = search.best_index_

    cv_r2 = (
        search.cv_results_[
            "mean_test_R2"
        ][best_index]
    )

    cv_mae = -(
        search.cv_results_[
            "mean_test_MAE"
        ][best_index]
    )

    cv_rmse = -(
        search.cv_results_[
            "mean_test_RMSE"
        ][best_index]
    )

    train_prediction = best_model.predict(
        X_train
    )

    test_prediction = best_model.predict(
        X_test
    )

    train_r2 = r2_score(
        y_train,
        train_prediction
    )

    train_mae = mean_absolute_error(
        y_train,
        train_prediction
    )

    train_rmse = np.sqrt(
        mean_squared_error(
            y_train,
            train_prediction
        )
    )

    test_r2 = r2_score(
        y_test,
        test_prediction
    )

    test_mae = mean_absolute_error(
        y_test,
        test_prediction
    )

    test_rmse = np.sqrt(
        mean_squared_error(
            y_test,
            test_prediction
        )
    )

    print(f"\n===== {model_name} =====")
    print("Best parameters:", search.best_params_)

    print(
        f"CV: R2={cv_r2:.6f} | "
        f"MAE={cv_mae:.6f} | "
        f"RMSE={cv_rmse:.6f}"
    )

    print(
        f"Train: R2={train_r2:.6f} | "
        f"MAE={train_mae:.6f} | "
        f"RMSE={train_rmse:.6f}"
    )

    print(
        f"Test: R2={test_r2:.6f} | "
        f"MAE={test_mae:.6f} | "
        f"RMSE={test_rmse:.6f}"
    )

    result = {
        "Model": model_name,
        "CV_R2": cv_r2,
        "CV_MAE": cv_mae,
        "CV_RMSE": cv_rmse,
        "Train_R2": train_r2,
        "Train_MAE": train_mae,
        "Train_RMSE": train_rmse,
        "Test_R2": test_r2,
        "Test_MAE": test_mae,
        "Test_RMSE": test_rmse,
        "Best_parameters": str(
            search.best_params_
        )
    }

    prediction_result = pd.DataFrame({
        GROUP_COL: test_data[GROUP_COL],
        "y_true": y_test,
        "y_pred": test_prediction
    })

    prediction_result.to_csv(
        f"{OUTPUT_DIR}/{model_name}_test_predictions.csv",
        index=False
    )

    return best_model, result


In [6]:
# ============================================================
# 7. XGBoost
# ============================================================

xgb_model = XGBRegressor(
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

xgb_grid = {
    "n_estimators": [
        100,
        200,
        300
    ],
    "max_depth": [
        2,
        3,
        4
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ]
}

best_xgb, xgb_result = train_and_evaluate(
    model_name="XGBoost",
    model=xgb_model,
    param_grid=xgb_grid
)



===== XGBoost =====
Best parameters: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200}
CV: R2=0.676807 | MAE=0.126573 | RMSE=0.204702
Train: R2=0.993099 | MAE=0.022798 | RMSE=0.031423
Test: R2=0.684217 | MAE=0.146302 | RMSE=0.237318


In [7]:
# ============================================================
# 8. Ridge
# ============================================================

ridge_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        Ridge()
    )
])

ridge_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ]
}

best_ridge, ridge_result = train_and_evaluate(
    model_name="Ridge",
    model=ridge_model,
    param_grid=ridge_grid
)



===== Ridge =====
Best parameters: {'model__alpha': 1}
CV: R2=0.614057 | MAE=0.151541 | RMSE=0.221722
Train: R2=0.825080 | MAE=0.120968 | RMSE=0.158206
Test: R2=0.769864 | MAE=0.157315 | RMSE=0.202595


In [8]:
# ============================================================
# 9. Kernel ridge
# ============================================================

krr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        KernelRidge(
            kernel="rbf"
        )
    )
])

krr_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10
    ],
    "model__gamma": [
        0.001,
        0.01,
        0.1,
        1
    ]
}

best_krr, krr_result = train_and_evaluate(
    model_name="Kernel Ridge",
    model=krr_model,
    param_grid=krr_grid
)



===== Kernel Ridge =====
Best parameters: {'model__alpha': 0.001, 'model__gamma': 0.01}
CV: R2=0.625751 | MAE=0.130997 | RMSE=0.212071
Train: R2=0.948298 | MAE=0.058580 | RMSE=0.086012
Test: R2=0.695894 | MAE=0.170131 | RMSE=0.232889


In [9]:
# ============================================================
# 10. Support vector regression
# ============================================================

svr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        SVR(
            kernel="rbf"
        )
    )
])

svr_grid = {
    "model__C": [
        0.1,
        1,
        10,
        100
    ],
    "model__epsilon": [
        0.01,
        0.05,
        0.10,
        0.20
    ],
    "model__gamma": [
        "scale",
        0.01,
        0.10,
        1.0
    ]
}

best_svr, svr_result = train_and_evaluate(
    model_name="SVR",
    model=svr_model,
    param_grid=svr_grid
)


===== SVR =====
Best parameters: {'model__C': 10, 'model__epsilon': 0.05, 'model__gamma': 'scale'}
CV: R2=0.745720 | MAE=0.122758 | RMSE=0.180228
Train: R2=0.933693 | MAE=0.063610 | RMSE=0.097406
Test: R2=0.790418 | MAE=0.140808 | RMSE=0.193337


In [10]:
# ============================================================
# 11. Gaussian process regression
# ============================================================

gpr_kernel = (
    ConstantKernel(
        constant_value=1.0,
        constant_value_bounds=(1e-2, 1e2)
    )
    * RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-2, 1e2)
    )
    + WhiteKernel(
        noise_level=1e-3,
        noise_level_bounds=(1e-6, 1e0)
    )
)

gpr_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        GaussianProcessRegressor(
            kernel=gpr_kernel,
            normalize_y=True,
            n_restarts_optimizer=5,
            random_state=RANDOM_STATE
        )
    )
])

gpr_grid = {
    "model__alpha": [
        1e-8,
        1e-6,
        1e-4,
        1e-2
    ]
}

best_gpr, gpr_result = train_and_evaluate(
    model_name="GPR",
    model=gpr_model,
    param_grid=gpr_grid
)


===== GPR =====
Best parameters: {'model__alpha': 1e-08}
CV: R2=0.764197 | MAE=0.122451 | RMSE=0.175755
Train: R2=0.942000 | MAE=0.061983 | RMSE=0.091100
Test: R2=0.782017 | MAE=0.145938 | RMSE=0.197173


In [11]:
# ============================================================
# 12. Random forest
# ============================================================

rf_model = RandomForestRegressor(
    random_state=RANDOM_STATE,
    n_jobs=1
)

rf_grid = {
    "n_estimators": [
        100,
        300,
        500
    ],
    "max_depth": [
        None,
        5,
        10
    ],
    "min_samples_leaf": [
        1,
        2,
        4
    ]
}

best_rf, rf_result = train_and_evaluate(
    model_name="Random Forest",
    model=rf_model,
    param_grid=rf_grid
)



===== Random Forest =====
Best parameters: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 300}
CV: R2=0.755738 | MAE=0.112707 | RMSE=0.180808
Train: R2=0.978844 | MAE=0.028704 | RMSE=0.055020
Test: R2=0.843661 | MAE=0.109581 | RMSE=0.166982


In [12]:
# ============================================================
# 13. Gradient boosting
# ============================================================

gbdt_model = GradientBoostingRegressor(
    random_state=RANDOM_STATE
)

gbdt_grid = {
    "n_estimators": [
        100,
        200,
        300
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ],
    "max_depth": [
        1,
        2,
        3
    ],
    "min_samples_leaf": [
        2,
        4,
        6
    ]
}

best_gbdt, gbdt_result = train_and_evaluate(
    model_name="Gradient Boosting",
    model=gbdt_model,
    param_grid=gbdt_grid
)


===== Gradient Boosting =====
Best parameters: {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 2, 'n_estimators': 300}
CV: R2=0.724408 | MAE=0.116999 | RMSE=0.192802
Train: R2=0.995190 | MAE=0.018543 | RMSE=0.026234
Test: R2=0.699078 | MAE=0.136337 | RMSE=0.231667


In [13]:
# ============================================================
# 14. TabPFN
# ============================================================

tabpfn_model = TabPFNRegressor(
    device="cuda",
    model_path=TABPFN_MODEL_PATH,
    random_state=RANDOM_STATE
)

tabpfn_grid = {}

best_tabpfn, tabpfn_result = train_and_evaluate(
    model_name="TabPFN",
    model=tabpfn_model,
    param_grid=tabpfn_grid,
    n_jobs=1
)


===== TabPFN =====
Best parameters: {}
CV: R2=0.807490 | MAE=0.109190 | RMSE=0.161118
Train: R2=0.996849 | MAE=0.011587 | RMSE=0.021233
Test: R2=0.866171 | MAE=0.114057 | RMSE=0.154494


In [14]:
# ============================================================
# 15. Model comparison
# ============================================================

comparison_results = pd.DataFrame([
    xgb_result,
    ridge_result,
    krr_result,
    svr_result,
    gpr_result,
    rf_result,
    gbdt_result,
    tabpfn_result
])

comparison_results = (
    comparison_results
    .sort_values(
        "CV_R2",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nFinal model comparison:")

print(
    comparison_results[
        [
            "Model",
            "CV_R2",
            "CV_MAE",
            "CV_RMSE",
            "Train_R2",
            "Train_MAE",
            "Train_RMSE",
            "Test_R2",
            "Test_MAE",
            "Test_RMSE"
        ]
    ]
)

comparison_results.to_csv(
    f"{OUTPUT_DIR}/model_comparison_results.csv",
    index=False
)



Final model comparison:
               Model     CV_R2    CV_MAE   CV_RMSE  Train_R2  Train_MAE  \
0             TabPFN  0.807490  0.109190  0.161118  0.996849   0.011587   
1                GPR  0.764197  0.122451  0.175755  0.942000   0.061983   
2      Random Forest  0.755738  0.112707  0.180808  0.978844   0.028704   
3                SVR  0.745720  0.122758  0.180228  0.933693   0.063610   
4  Gradient Boosting  0.724408  0.116999  0.192802  0.995190   0.018543   
5            XGBoost  0.676807  0.126573  0.204702  0.993099   0.022798   
6       Kernel Ridge  0.625751  0.130997  0.212071  0.948298   0.058580   
7              Ridge  0.614057  0.151541  0.221722  0.825080   0.120968   

   Train_RMSE   Test_R2  Test_MAE  Test_RMSE  
0    0.021233  0.866171  0.114057   0.154494  
1    0.091100  0.782017  0.145938   0.197173  
2    0.055020  0.843661  0.109581   0.166982  
3    0.097406  0.790418  0.140808   0.193337  
4    0.026234  0.699078  0.136337   0.231667  
5    0.031423  0.